In [ ]:
def generate_row(alternative_id: str, variable: str | tuple[str, str], ignore_configuration: bool = False) -> str:
    
    logger.info(f'Generating row for {alternative_id} and variable {variable}')
    
    alternative = alternatives[alternative_id]
    bold = alternative.get('bold', False)
    query_idx = get_idx(alternatives, system, alternative_id)

    metric_values = get_metric_values(
        tr_results_path=results_path / df_info.loc[query_idx, 'training_results_filename'],
        val_results_path=results_path / df_info.loc[query_idx, 'validation_results_filename'],
        tr_data_path=data_path / df_info.loc[query_idx, 'training_data_filename'],
        val_data_path=data_path / df_info.loc[query_idx, 'validation_data_filename'],
        variable=variable
    )

    elapsed_time_val = df_info.loc[query_idx, 'elapsed_time_validation']

    topology = df_info.loc[query_idx, 'topology']
    if not pd.isna(topology):
        # Find a comma, if there is one, if variable is "Tout", take the text before the comma, if not, take the text after the comma
        # First check if there is a comma, otherwise continue
        if ',' in topology:
            topology = topology.split(',')[0] if variable == "Tout" else topology.split(',')[1]
    else:
        topology = "N/A"
        
    configuration = alternative['configuration'] if not ignore_configuration else "-"

    if bold:
        row_str = f" & & \\textbf{{{alternative['name']}}} & & {configuration} & & {topology}"
        row_str +=  "".join([f" & & {m:.2f}" if i%2==0 else f" & & \\textbf{{{m:.2f}}}" for i, m in enumerate(metric_values)])
    else:
        row_str = f" & & {alternative['name']} & & {configuration} & & {topology}"
        row_str += "".join([f" & & {m:.2f}" for m in metric_values])
    row_str += rf" & & {elapsed_time_val:.3f} & \\" # One space and another for manually written evaluation time
    
    logger.info(f'Generated row for {alternative}')
    
    return row_str


In [ ]:

string_inicial = r"""\begin{landscape}
\begin{table}[]
\centering
\caption{Summary table of the prediction results obtained with the different modelling approaches studied.}
\label{tab:results}
\resizebox{\textwidth}{!}{%
    """
    
    string_final = r"""}
\end{table}
\end{landscape}
"""

table = rf"""
\begin{{tabular}}{{cclccccccccccccccccccccccc}}
\hline
\multicolumn{{1}}{{c}}{{\multirow{{3}}{{*}}{{\textbf{{\begin{{tabular}}[c]{{@{{}}c@{{}}}}Predicted\\ variable\end{{tabular}}}}}}}} & &
\multicolumn{{1}}{{c}}{{\multirow{{3}}{{*}}{{\textbf{{\begin{{tabular}}[c]{{@{{}}c@{{}}}}Modelling\\ alternative\end{{tabular}}}}}}}} & &
\multicolumn{{1}}{{c}}{{\multirow{{3}}{{*}}{{\textbf{{\begin{{tabular}}[c]{{@{{}}c@{{}}}}Model\\ config\end{{tabular}}}}}}}} & &
\multicolumn{{1}}{{c}}{{\multirow{{3}}{{*}}{{\textbf{{\begin{{tabular}}[c]{{@{{}}c@{{}}}}Topology\end{{tabular}}}}}}}} & &
\multicolumn{{15}}{{c}}{{\textbf{{Performance metric}}}} & &
\multicolumn{{1}}{{c}}{{\multirow{{3}}{{*}}{{\textbf{{\begin{{tabular}}[c]{{@{{}}c@{{}}}}Evaluation\\ time (s)\end{{tabular}}}}}}}} \\ 
\cline{{9-23}} \multicolumn{{1}}{{c}}{{}} & & \multicolumn{{1}}{{c}}{{}} & & & & & &
\multicolumn{{3}}{{c}}{{\textbf{{\begin{{tabular}}[c]{{@{{}}c@{{}}}}R$^2$\\ (-)\end{{tabular}}}}}} & &
\multicolumn{{3}}{{c}}{{\textbf{{\begin{{tabular}}[c]{{@{{}}c@{{}}}}RMSE\\ (s.u.)\end{{tabular}}}}}} & &
\multicolumn{{3}}{{c}}{{\textbf{{\begin{{tabular}}[c]{{@{{}}c@{{}}}}MAE\\ (s.u.)\end{{tabular}}}}}} & &
\multicolumn{{3}}{{c}}{{\textbf{{\begin{{tabular}}[c]{{@{{}}c@{{}}}}MAPE \\ (\%)\end{{tabular}}}}}} & &
\multicolumn{{1}}{{c}}{{}} \\
\cline{{9-11}} \cline{{13-15}} \cline{{17-19}} \cline{{21-23}}
\multicolumn{{1}}{{c}}{{}}  & & \multicolumn{{1}}{{c}}{{}}  & & \multicolumn{{1}}{{c}}{{}}  & & \multicolumn{{1}}{{c}}{{}}  & &
\multicolumn{{1}}{{c}}{{T}} & & \multicolumn{{1}}{{c}}{{V}} & &
\multicolumn{{1}}{{c}}{{T}} & & \multicolumn{{1}}{{c}}{{V}} & &
\multicolumn{{1}}{{c}}{{T}} & & \multicolumn{{1}}{{c}}{{V}} & &
\multicolumn{{1}}{{c}}{{T}} & & \multicolumn{{1}}{{c}}{{V}} & & \multicolumn{{1}}{{c}}{{}} \\
\cline{{1-1}} \cline{{3-3}} \cline{{5-5}} \cline{{7-7}} \cline{{9-9}} \cline{{11-11}} \cline{{13-13}} \cline{{15-15}} \cline{{17-17}} \cline{{19-19}} \cline{{21-21}} \cline{{23-23}} \cline{{25-25}}
\multirow{{{len(alternatives)+1}}}{{*}}{{T$_{{wct,out}}$ ($^\circ$C)}}
    % Include Poppe manually
    & & \textbf{{Physical model}} & & - & & - & & - & & \textbf{{{mv_p_t[0]:.2f}}} & & - & & \textbf{{{mv_p_t[1]:.2f}}} & & - & & \textbf{{{mv_p_t[2]:.2f}}} & & - & & \textbf{{{mv_p_t[3]:.2f}}} & & 6.288 & \\
    {"".join([generate_row(alternative, "Tout") for alternative in alternatives.keys()])} \hline
\multirow{{{len(alternatives)+1}}}{{*}}{{C$_{{w}}$ (l/h)}}
    % Include Poppe manually
    & & \textbf{{Physical model}} & & - & & - & & - & &  \textbf{{{mv_p_m[0]:.2f}}} & & - & & \textbf{{{mv_p_m[1]:.2f}}} & & - & & \textbf{{{mv_p_m[2]:.2f}}} & & - & & \textbf{{{mv_p_m[3]:.2f}}} & & 6.288 & \\
    {"".join([generate_row(alternative, ("m_w_lost", "Mlost")) for alternative in alternatives.keys()])} \hline
\end{{tabular}}%
"""

table = string_inicial + table + string_final
print(table)


In [ ]:
def generate_latex_table(
    var_ids: list[str], 
    ptops: list[OperationPoint],
    var_title: str = "Variables",
    fields_to_look_for: list[str] = ["signals", "extra_signals", "energetic_metrics", "exergetic_metrics", "separation_metrics"],
    val_keys: list[str] = ["value_uu", "mean_value", "value"],
    separated_rows: list[int] = None,
    only_variables: bool = False,
) -> str:
    
    cols = 2 if only_variables else 6
    
    var_units = []
    var_labels = []
    for var_id in var_ids:
        # Find the object in which the variable is stored
        for key in fields_to_look_for:
            if hasattr(ptops[0], key) and var_id in getattr(ptops[0], key):
                var_data = getattr(ptops[0], key)[var_id]
                if var_data.units:
                    var_unit = unit_symbol_to_latex[var_data.units] 
                elif var_id in metrics_info and metrics_info[var_id].get("units_latex"):
                    var_unit = metrics_info[var_id]["units_latex"]
                else:
                    var_unit = "-"
                var_units.append(var_unit)
        
                # print(f"{metrics_info[var_id]=}")
        
                if hasattr(var_data, "label_latex") and var_data.label_latex is not None:
                    var_labels.append(var_data.label_latex)
                elif vars_config.get(var_id) and vars_config[var_id].get("label_latex"):
                    var_labels.append(vars_config[var_id]["label_latex"])
                else:
                    var_labels.append(var_id)
                
                # Chapuza, la etiqueta ya debería estar formateada correctamente
                if "eta" in var_labels[-1] and "$" not in var_labels[-1]:
                    logger.warning(f"Variable '{var_id}' has eta in its label but is not formatted with $ signs: {var_labels[-1]}. Fixing it.")
                    # print(f"Found variable '{var_id}' in {key} with unit '{var_unit}' and label '{var_labels[-1]}'")
                    var_labels[-1] = f"${var_labels[-1]}$"
                
                # print(f"Found variable '{var_id}' in {key} with unit '{var_unit}' and label '{var_labels[-1]}'")
                    
                break
        else:
            raise ValueError(f"Variable '{var_id}' not found in any of the fields: {fields_to_look_for} of ptop.")
                     

    # Build header rows
    if only_variables:
        header_row1 = r"\multirow{2}{*}{} &  & \multicolumn{" + str(2*len(var_ids)) + r"}{c}{\textbf{" + var_title + "}} \\\\ \\cline{" + str(cols+1) + "-" + str(cols + 2*len(var_ids)) + "}"
        header_row2 = "           &  & "
    else:
        header_row1 = r"\multirow{2}{*}{} &  & \multirow{2}{*}{\begin{tabular}[c]{@{}c@{}}\textbf{Test date}\\ (UTC)\end{tabular}} &  & \multirow{2}{*}{\begin{tabular}[c]{@{}c@{}}\textbf{D}\\ (min)\end{tabular}} &  & \multicolumn{" + str(2*len(var_ids)) + r"}{c}{\textbf{" + var_title + "}} \\\\ \cline{" + str(cols+1) + "-" + str(cols + 2*len(var_ids)) + "}"
        header_row2 = "           &  &                 &  &                           &  & "
    
    header_row2 += " &  & ".join(
        [rf"\begin{{tabular}}[c]{{@{{}}c@{{}}}}{var_id}\\ {{\footnotesize ({var_unit}) }}\end{{tabular}}" for var_id, var_unit in zip(var_labels, var_units)]
    )
    
    if only_variables:
        header_row2 += r" \\ " + " ".join([rf"\cline{{{cols+1 + 2*i}-{cols+1 + 2*i}}}" for i in range(len(var_ids))])
    else:
        header_row2 += r" \\ \cline{3-3} \cline{5-5}" + " ".join([rf"\cline{{{cols+1 + 2*i}-{cols+1 + 2*i}}}" for i in range(len(var_ids))])

    # Header formatting
    table_lines = [
        r"\begin{table}[]",
        r"\begin{tabular}{" + "c" * (cols+1 + 2*len(var_ids)) + "}",
        r"\hline",
        header_row1,
        header_row2,
    ]

    # Add data rows
    for ptop_idx, ptop in enumerate(ptops):
        
        if separated_rows and ptop_idx in separated_rows:
            table_lines.append(rf"\cline{{3-{cols + 2*len(var_ids)}}}")
        
        date_str = ptop.start_time.strftime("%Y%m%d %H:%M")
        if only_variables:
            row = [f"{ptop_idx+1}", " "]            
        else:
            row = [f"{ptop_idx+1}", " ", date_str, " ", str(int(ptop.duration/60)), " "]
            
        for var_id in var_ids:
            
            # Find the object in which the variable is stored
            for key in fields_to_look_for:
                if hasattr(ptop, key) and var_id in getattr(ptop, key):
                    var_data = getattr(ptop, key)[var_id]
                    for val_key in val_keys:
                        if val_key in var_data.__dict__ and getattr(var_data, val_key) is not None:
                            var_val = getattr(var_data, val_key)
                            break
                    else:
                        raise ValueError(f"Variable '{var_id}' not found or None in {key} of ptop {ptop.id} ({ptop_idx=}).")
            
            # print(var_val)
            if val_key == "value_uu":
                row.append(f"${var_val:.1uL}$")
            else:
                row.append(f"{var_val:.2f}")
            row.append(" ")  # empty spacer column
        # print(row)
        table_lines.append(" & ".join(row) + r" \\")

    # Final line
    table_lines.append(r"\hline")
    table_lines.append(r"\end{tabular}")
    table_lines.append(r"\end{table}")

    return "\n".join(table_lines)


separated_rows = [2, 5]
var_ids = ["Ts_in", "Tc_out", "Ms", "Mf", "Md", "Ts_out", "Tc_in", "wf", "wd", "Mc", "J"] #, "Pvc", "Pv1", "Ld", "Lb"]

print( generate_latex_table(var_ids, ptops, var_title="Measured variables",separated_rows=separated_rows, only_variables=True) )
